# Ray RLlib quickstart: PPO on Taxi-v3

This notebook uses the **current RLlib API** (`env_runners`, RLModules, connectors).

**Setup (once):**

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Then select that kernel in Jupyter / VS Code / Cursor. Prefer installing from `requirements.txt` over `%pip` so the environment matches the repo pins.

Same workflow as a script: `python train_taxi_ppo.py`.

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*RLModule\(config=\[RLModuleConfig object\]\).*",
    category=DeprecationWarning,
)

from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.connectors.env_to_module import FlattenObservations
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig

config = (
    PPOConfig()
    .environment("Taxi-v3")
    .env_runners(
        num_env_runners=2,
        # Taxi observations are discrete ints; one-hot flatten for the MLP.
        # Signature must be (env, spaces, device) — args may be unused/None.
        env_to_module_connector=lambda env, spaces, device: FlattenObservations(),
    )
    .rl_module(model_config=DefaultModelConfig(fcnet_hiddens=[64, 64]))
    # Dedicated eval EnvRunner; call algo.evaluate() manually (no auto interval).
    .evaluation(evaluation_num_env_runners=1)
    .debugging(log_level="ERROR")
)

algo = config.build_algo()

try:
    for i in range(1, 6):
        result = algo.train()
        ret = result["env_runners"]["episode_return_mean"]
        steps = result["num_env_steps_sampled_lifetime"]
        print(f"iter={i}  episode_return_mean={ret:.1f}  env_steps={steps}")

    eval_result = algo.evaluate()
    eval_ret = eval_result["env_runners"]["episode_return_mean"]
    print(f"evaluate  episode_return_mean={eval_ret:.1f}")
finally:
    # Release EnvRunner / Learner actors.
    algo.stop()


## What changed vs older RLlib notebooks

| Old (Ray ~2.3) | Current |
| --- | --- |
| `.rollouts(num_rollout_workers=…)` | `.env_runners(num_env_runners=…)` |
| `.framework("torch")` + `model={…}` | Torch-first RLModules + `DefaultModelConfig` |
| `config.build()` | `config.build_algo()` |
| Discrete Taxi obs as raw ints | `FlattenObservations` connector |
| Leave actors running | Call `algo.stop()` when finished |